https://www.kaggle.com/competitions/face-matching-aicc-round-2/overview

In [1]:
!pip install transformers

In [34]:
import os
import pandas as pd
import cv2
from tqdm import tqdm
import numpy as np
from transformers import CLIPProcessor, CLIPModel

In [35]:
# edit with your dataset path
root_path = "/kaggle/input/face-matching-aicc-round-2/face-matching"

In [42]:
processor = CLIPProcessor.from_pretrained("laion/CLIP-ViT-H-14-laion2B-s32B-b79K")
model = CLIPModel.from_pretrained('laion/CLIP-ViT-H-14-laion2B-s32B-b79K')

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.94G [00:00<?, ?B/s]

In [43]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 1024)
      (position_embedding): Embedding(77, 1024)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-23): 24 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          )
          (layer_norm2): LayerNorm((1024,), e

# Dataset

In [44]:
ref_df = pd.read_csv(f"{root_path}/ref_img.csv", dtype={'ref_img': str})
ref_ids = ref_df["ref_img"].tolist()

img_dir = f"{root_path}/images"
all_images = os.listdir(img_dir)

In [45]:
features = {}
for img_path in tqdm(all_images):
    img_id = img_path[:-len(".jpg")]
    img_path = f"{img_dir}/{img_path}"

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    inputs = processor(images = img, return_tensors = 'pt')
    inputs = inputs.to(device)
    with torch.no_grad():
        img = model.get_image_features(**inputs)
    features[img_id] = img.squeeze()

100%|██████████| 109/109 [00:16<00:00,  6.43it/s]


In [46]:
from torch.nn.functional import cosine_similarity

# Submission

In [47]:
results = []
for ref_id in ref_ids:
    ref_feature = features[ref_id]

    # compute distances to all images
    distances = {}
    for img_id, feature in features.items():
        dist = 1 - cosine_similarity(ref_feature.unsqueeze(0),feature.unsqueeze(0))
        distances[img_id] = dist

    # sort by distance, exclude reference, take top 5
    sorted_ids = sorted(distances.items(), key=lambda x: x[1])
    top_5 = [img_id for img_id, _ in sorted_ids if img_id != ref_id][:5]

    results.append({"ref_img": ref_id, "photos": "|".join(top_5)})

In [48]:
submission = pd.DataFrame(results)
submission

,ref_img,photos
0,048,101|102|078|066|068
1,025,067|041|083|033|006
2,095,021|014|047|097|107
3,043,027|034|013|023|081
4,105,024|029|010|093|063
5,071,011|049|065|088|070
6,046,060|019|052|010|107
7,096,098|068|015|018|035
8,020,053|077|091|028|008
9,085,036|009|037|106|103


In [49]:
submission.to_csv("submission.csv", index=False)